# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [ ]:
#!pip install numpy 
#!pip install pandas
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


In [2]:
recipes = pd.read_csv('recipes.tsv', sep="\t")
recipes = recipes.set_index("recipe_slug")
recipes

,recipe_title,recipe_introduction
recipe_slug,,
falafel,Falafel,Falafel is a popular Middle Eastern dish made ...
spamburger,Spamburger,Spamburger is a type of hamburger that is made...
bacon-fried-rice,Bacon Fried Rice,Bacon fried rice is a savory and satisfying di...
chicken-fingers,Chicken Fingers,Chicken fingers are a popular dish made from c...
apple-crisp,Apple Crisp,Apple crisp is a classic dessert made with bak...
...,...,...
bacon-mac-and-cheese,Bacon Mac And Cheese,Bacon mac and cheese is a delicious and comfor...
chicken-alfredo-lasagna,Chicken Alfredo Lasagna,Chicken alfredo lasagna is a delicious twist o...
classic-beef-lasagna,Classic Beef Lasagna,Classic beef lasagna is a hearty and comfortin...


In [21]:
ratings = pd.DataFrame(index=recipes.index)
ratings["rating_raw"] = 3 #placeholder number??? 3??
ratings

,rating_raw
recipe_slug,
falafel,3
spamburger,3
bacon-fried-rice,3
chicken-fingers,3
apple-crisp,3
...,...
bacon-mac-and-cheese,3
chicken-alfredo-lasagna,3
classic-beef-lasagna,3


In [ ]:
#Persona: Vegetarian who loves cheese 

text = (recipes["recipe_title"].astype(str) + " " + recipes["recipe_introduction"].astype(str)).str.lower()

meat_items = text.str.contains("bacon|chicken|beef|pork|ham|sausage|turkey|spam", regex=True)
veg_items = text.str.contains("vegetarian|veggie|spinach|mushroom|falafel|tofu|lentil|bean|chickpea|eggplant", regex=True)
cheese_items = text.str.contains("cheese|cheddar|mozzarella|parmesan|ricotta|feta|gouda|cream|alfredo", regex=True)

ratings.loc[meat_items, "rating_raw"] = 1 #bad 
ratings.loc[veg_items & (ratings["rating_raw"] != 1), "rating_raw"] = 4 #good
ratings.loc[cheese_items & (ratings["rating_raw"] != 1), "rating_raw"] = 5 #best!!

ratings["rating_raw"].value_counts()


rating_raw
1    36
5    35
3    26
4     3
Name: count, dtype: int64

In [5]:
ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25

ratings_out = ratings.reset_index()[["recipe_slug", "rating"]]


In [6]:
ratings_out.to_csv("ratings.tsv", sep="\t", index=False)

ratings_out.head()

,recipe_slug,rating
0,falafel,0.75
1,spamburger,0.00
2,bacon-fried-rice,0.00
3,chicken-fingers,0.00
4,apple-crisp,1.00


In [7]:
ratings_out["rating"].min(), ratings_out["rating"].max()
ratings_out.shape


(100, 2)

Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [8]:
# YOUR CHANGES HERE

# Load ratings
ratings = pd.read_csv("ratings.tsv", sep="\t")
ratings = ratings.set_index("recipe_slug")

# Load recipe tags
recipe_tags = pd.read_csv("recipe-tags.tsv", sep="\t")

# Start features using same order as ratings
features = pd.DataFrame(index=ratings.index)

In [9]:

# Bias column
features["bias"] = 1

# Helper function
def add_tag_feature(tag_name):
    slugs_with_tag = recipe_tags.loc[
        recipe_tags["recipe_tag"] == tag_name,
        "recipe_slug"]
    features[tag_name] = features.index.isin(slugs_with_tag).astype(int)

# Add all tag columns
all_tags = sorted(recipe_tags["recipe_tag"].unique())

for tag in all_tags:
    add_tag_feature(tag)

/tmp/ipykernel_2059/2886117765.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[tag_name] = features.index.isin(slugs_with_tag).astype(int)
/tmp/ipykernel_2059/2886117765.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[tag_name] = features.index.isin(slugs_with_tag).astype(int)
/tmp/ipykernel_2059/2886117765.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.

In [10]:
features_out = features.reset_index()
features_out.to_csv("features.tsv", sep="\t", index=False)

features_out.head()


/tmp/ipykernel_2059/1434064471.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features_out = features.reset_index()


,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,1,0,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0


Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [11]:
# YOUR CHANGES HERE
# 1) Load files
ratings = pd.read_csv("ratings.tsv", sep="\t").set_index("recipe_slug")      # target y
features = pd.read_csv("features.tsv", sep="\t").set_index("recipe_slug")    # inputs X

# 2) Make sure they are aligned in the same order
features = features.reindex(ratings.index)

In [12]:
# 3) Split into X and y
X = features.values
y = ratings["rating"].values

# 4) Fit ridge regression model (alpha=1)
model = Ridge(alpha=1, fit_intercept=False)
model.fit(X, y)

# 5) View coefficients (feature weights)
coefs = pd.Series(model.coef_, index=features.columns).sort_values(ascending=False)
coefs.head(15), coefs.tail(15)


(vegetarian           0.342403
 bias                 0.332675
 pickledvegetables    0.224530
 baking               0.203057
 summer               0.170428
 onion                0.154383
 easydinner           0.149425
 mexicancuisine       0.141144
 crisp                0.138446
 meat                 0.126062
 customizable         0.125515
 handheld             0.125515
 dessert              0.124865
 mexican              0.120709
 baked                0.116992
 dtype: float64,
 pastasauce       -0.097357
 cranberry        -0.097995
 fresh            -0.099775
 blueberry        -0.099981
 chicken          -0.106594
 sichuancuisine   -0.108112
 italiancuisine   -0.114797
 alfredo          -0.114797
 partyfood        -0.121716
 casserole        -0.136459
 vegan            -0.145739
 groundbeef       -0.187936
 tomatosauce      -0.242702
 nachos           -0.260216
 bacon            -0.356241
 dtype: float64)

Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [13]:
# YOUR CHANGES HERE
# After model.fit(X, y)
coefs = pd.Series(model.coef_, index=features.columns)

model_out = coefs.reset_index()
model_out.columns = ["recipe_tag", "coefficient"]

model_out.to_csv("model.tsv", sep="\t", index=False)

model_out.head()

,recipe_tag,coefficient
0,bias,0.332675
1,alfredo,-0.114797
2,almond,0.021375
3,american,0.022515
4,appetizer,0.039007


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [14]:
# YOUR CHANGES HERE

# Load
ratings = pd.read_csv("ratings.tsv", sep="\t").set_index("recipe_slug")
features = pd.read_csv("features.tsv", sep="\t").set_index("recipe_slug")

# Align
features = features.reindex(ratings.index)

In [19]:
X = features.values
y = ratings["rating"].values

# Fit ridge (alpha=1)
alpha = 1
model = Ridge(alpha=alpha, fit_intercept=False)
model.fit(X, y)

# Point estimates for each recipe (predicted rating)
y_hat = model.predict(X)



In [20]:
estimates_df = pd.DataFrame({
    "recipe_slug": features.index,
    "score_estimate": y_hat
})

estimates_df.to_csv("estimates.tsv", sep="\t", index=False)

estimates_df.head()


,recipe_slug,score_estimate
0,falafel,0.755371
1,spamburger,0.026512
2,bacon-fried-rice,0.022228
3,chicken-fingers,0.033470
4,apple-crisp,0.956218


Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [ ]:
# YOUR CHANGES HERE

# Load + align
ratings = pd.read_csv("ratings.tsv", sep="\t").set_index("recipe_slug")
features = pd.read_csv("features.tsv", sep="\t").set_index("recipe_slug")
features = features.reindex(ratings.index)

X = features.values
y = ratings["rating"].values

# Parameters
lam = 1.0
alpha = 2.0

# A and b
d = X.shape[1]
A = lam * np.eye(d)
b = np.zeros(d)

# Try every recipe once   #update: A += x x^T, b += x * reward
for i in range(len(X)):
    x = X[i]
    r = y[i]
    A += np.outer(x, x)
    b += x * r

A_inv = np.linalg.inv(A)
theta_hat = A_inv @ b

def calculate_bound(x):
    expected_reward = theta_hat @ x
    variance = x @ A_inv @ x
    return expected_reward + alpha * np.sqrt(variance)

score_bound = np.array([calculate_bound(x) for x in X])

bounds = pd.DataFrame({
    "recipe_slug": features.index,
    "score_bound": score_bound
})

bounds.to_csv("bounds.tsv", sep="\t", index=False)
bounds.head()


,recipe_slug,score_bound
0,falafel,2.521627
1,spamburger,1.920848
2,bacon-fried-rice,1.888385
3,chicken-fingers,1.827585
4,apple-crisp,2.732560


Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [17]:
# YOUR CHANGES HERE

# Load + align
ratings = pd.read_csv("ratings.tsv", sep="\t").set_index("recipe_slug")
features = pd.read_csv("features.tsv", sep="\t").set_index("recipe_slug")
features = features.reindex(ratings.index)

In [18]:
X = features.values
slugs = features.index.to_numpy()
reward_lookup = ratings["rating"].to_dict()

# Parameters (same as Part 5)
lam = 1.0
alpha = 2.0

# Start with no data
d = X.shape[1]
A = lam * np.eye(d)
b = np.zeros(d)

def calculate_bounds(A, b, X, alpha):
    A_inv = np.linalg.inv(A)
    theta_hat = A_inv @ b
    est = X @ theta_hat
    var = np.sum((X @ A_inv) * X, axis=1)
    return est + alpha * np.sqrt(var)

history = []

T = 100
for t in range(T):
    # 1) compute bounds for all recipes
    bounds = calculate_bounds(A, b, X, alpha)

    # 2) choose the best
    i = int(np.argmax(bounds))
    chosen_slug = slugs[i]
    chosen_bound = float(bounds[i])

    # 3) observe reward from Part 1 ratings
    reward = float(reward_lookup[chosen_slug])

    # 4) record
    history.append({
        "recipe_slug": chosen_slug,
        "score_bound": chosen_bound,
        "reward": reward
    })

    # 5) update A and b with (x, reward)
    x = X[i]
    A += np.outer(x, x)
    b += x * reward

recommendations = pd.DataFrame(history)
recommendations.to_csv("recommendations.tsv", sep="\t", index=False)

recommendations.head()


,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,1.0
1,ramen,7.270093,0.0
2,quesadillas,7.235617,1.0
3,ma-la-chicken,7.226414,0.0
4,spamburger,6.990261,0.0


Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

In [22]:
text = """I used the lecture notes, in-class examples, and provided course notebooks as references for implementing ridge regression and the LinUCB algorithm. No additional external libraries beyond those covered in the module (pandas, numpy, and scikit-learn) were used.
"""

with open("acknowledgements.txt", "w") as f:
    f.write(text)

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.